In [2]:
import pickle
from pytube import YouTube
import requests
import cv2
import os
import random
import json

with open(r"extract_vidlinks_from_json\videos.pkl", "rb") as file:
    vid_links = pickle.load(file)

In [ ]:
with open("links.json", "w") as file:
    json.dump({"video": vid_links}, file, indent=2)

In [ ]:
# File path to the video
video_path = "[Abyss 24][PMA 12-1-1] Yoimiya Monopyro - 247s.mp4"

# Open the video file
video = cv2.VideoCapture(video_path)

# Frame number you want to get the timestamp for
frame_number = 100  # Replace this with the desired frame number

# Set the frame position to the desired frame number
video.set(cv2.CAP_PROP_POS_FRAMES, frame_number)

# Get the frame rate of the video
frame_rate = video.get(cv2.CAP_PROP_FPS)

# Calculate the timestamp of the specific frame
timestamp = frame_number / frame_rate

# Release the video object
video.release()

# Display the timestamp
print(f"Timestamp of frame {frame_number}: {timestamp} seconds")

In [ ]:
###############
###############
###############
# Get video frame
###############
###############
###############

# File path to the video
video_path = "[Abyss 24][PMA 12-1-1] Yoimiya Monopyro - 247s.mp4"

# Open the video file
video = cv2.VideoCapture(video_path)

# Time position you want to extract the frame from (in seconds)
time_position = timestamp  # Replace this with the desired time position

# Get the frame rate of the video
frame_rate = video.get(cv2.CAP_PROP_FPS)

# Calculate the frame number corresponding to the time position
frame_number = int(time_position * frame_rate)

# Set the frame position to the calculated frame number
video.set(cv2.CAP_PROP_POS_FRAMES, frame_number)

# Read the frame at the specified frame number
ret, frame = video.read()

if ret:
    # Save the frame as an image (adjust the file name as needed)
    cv2.imwrite("exported_frame.jpg", frame)

# Release the video object
video.release()

In [ ]:
def get_convert_size(file_path):
    size_bytes = os.path.getsize(file_path)
    # Convert bytes to a more human-readable format
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if size_bytes < 1024.0:
            return f"{size_bytes:.2f} {unit}"
        size_bytes /= 1024.0
    return size_bytes

In [ ]:
yt = YouTube("https://www.youtube.com/watch?v=7TakZUJoRjE")
x = yt.streams.get_by_itag(397)
x.download()

In [3]:
response = requests.get(
    "https://rr5---sn-q4fzen7y.googlevideo.com/videoplayback?expire=1717273764&ei=RDBbZt7BE9n4sfIP79ma6AY&ip=35.86.203.31&id=o-ACFCm-Fl3zL76cb8DCPuN3s2IdT1Pl0BUxOecKNkoV0O&itag=299&source=youtube&requiressl=yes&xpc=EgVo2aDSNQ%3D%3D&vprv=1&svpuc=1&mime=video%2Fmp4&rqh=1&gir=yes&clen=11998893&dur=21.649&lmt=1717223004677428&keepalive=yes&c=IOS&txp=6219224&sparams=expire%2Cei%2Cip%2Cid%2Citag%2Csource%2Crequiressl%2Cxpc%2Cvprv%2Csvpuc%2Cmime%2Crqh%2Cgir%2Cclen%2Cdur%2Clmt&sig=AJfQdSswRAIgGg22v5qjq2Wa-7yMV1wpnEqzDhq-PcMtK8_ItmCvYsgCIBclKxGSiK3FaUwbwgB98SH-8fm8XUnx8FECPGQQJRsc&redirect_counter=1&cm2rm=sn-nx5zz76&fexp=24350477&req_id=9928a3bb053aa3ee&cms_redirect=yes&cmsv=e&mh=Wa&mip=14.226.249.141&mm=34&mn=sn-q4fzen7y&ms=ltu&mt=1717251784&mv=D&mvi=5&pl=0&lsparams=mh,mip,mm,mn,ms,mv,mvi,pl&lsig=AHlkHjAwRAIgeG5VQmqG1O4lEMt1YoRI4PE2kl6SejxFxtR7cd_BPOICIE4X0F6l3aa7PKlQfrfctVZo8uXW3aeqA5hGXSkZa88P"
)
with open("video.mp4", "wb") as video_file:
    for chunk in response.iter_content():
        video_file.write(chunk)

In [ ]:
links = random.sample(vid_links, 20)
for i in links:
    print(i)

In [4]:
# Allow direct execution
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import yt_dlp
import yt_dlp.options

create_parser = yt_dlp.options.create_parser


def parse_patched_options(opts):
    patched_parser = create_parser()
    patched_parser.defaults.update(
        {
            "ignoreerrors": False,
            "retries": 0,
            "fragment_retries": 0,
            "extract_flat": False,
            "concat_playlist": "never",
        }
    )
    yt_dlp.options.create_parser = lambda: patched_parser
    try:
        return yt_dlp.parse_options(opts)
    finally:
        yt_dlp.options.create_parser = create_parser


default_opts = parse_patched_options([]).ydl_opts


def cli_to_api(opts, cli_defaults=False):
    opts = (yt_dlp.parse_options if cli_defaults else parse_patched_options)(
        opts
    ).ydl_opts

    diff = {k: v for k, v in opts.items() if default_opts[k] != v}
    if "postprocessors" in diff:
        diff["postprocessors"] = [
            pp
            for pp in diff["postprocessors"]
            if pp not in default_opts["postprocessors"]
        ]
    return diff


if __name__ == "__main__":
    from pprint import pprint

    print("\nThe arguments passed translate to:\n")
    pprint(cli_to_api(sys.argv[1:]))
    print("\nCombining these with the CLI defaults gives:\n")
    pprint(cli_to_api(sys.argv[1:], True))

In [2]:
# pip install yt_dlp
# pip install https://github.com/seproDev/yt-dlp-ChromeCookieUnlock/archive/main.zip
from yt_dlp.utils import download_range_func
import yt_dlp
import json
import subprocess
import os

final_filename = ""

def yt_dlp_monitor(d):
    global final_filename
    if d['status'] == 'finished':
        final_filename = d['filename']
        print(f"Download finished, saved to: {final_filename}")

def get_webm_files(directory):
    return [file for file in os.listdir(directory) if file.endswith(".webm")]

ydl_opts = {
    "paths": {"home": "downloadedvideos/"},
    # "format_sort": ["aext:m4a"],
    'format': 'ba',
    "outtmpl": "%(title)s [%(section_start)s-%(section_end)s].%(ext)s",
    "overwrite": True,
    # 'proxy': '14.226.227.172:10000',
    # 'listformats':True,
    # 'cookiesfrombrowser':('edge',),
    # 'force_keyframes_at_cuts': True,
    'ffmpeg_location': r'C:\Users\HP\ffmpeg-2024-05-15-git-7b47099bc0-essentials_build\bin',
    # 'progress_hooks': [yt_dlp_monitor],
    # "download_ranges": download_range_func(None, [(0, 230)]),
    # 'cookiefile':cookies,
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    # print(json.dumps(ydl.extract_info('https://www.youtube.com/watch?v=15OFObgaDUg', download=False),indent=4))
    ydl.download(["https://www.youtube.com/watch?v=scuB4o9rHX8"])
    # print(f"Final filename: {final_filename}")
    files = get_webm_files("downloadedvideos")
    for i in files:
        filename = i.split(".")[0]
        try:
            # Construct the FFmpeg command
            command = [
                "ffmpeg", "-i", f'"{filename}.webm"', "-vn", "-ab", "192k", "-ar", "44100", "-y", f'"{filename}.mp3"'
            ]
            subprocess.run(command, shell=True, check=True)
            print(f"Conversion successful: {filename}")
        except subprocess.CalledProcessError as e:
            print(f"Error during conversion: {e}")



[youtube] Extracting URL: https://www.youtube.com/watch?v=scuB4o9rHX8
[youtube] scuB4o9rHX8: Downloading webpage
[youtube] scuB4o9rHX8: Downloading tv client config
[youtube] scuB4o9rHX8: Downloading player 9a279502
[youtube] scuB4o9rHX8: Downloading tv player API JSON
[youtube] scuB4o9rHX8: Downloading ios player API JSON
[youtube] scuB4o9rHX8: Downloading m3u8 information
[info] scuB4o9rHX8: Downloading 1 format(s): 251
[download] Destination: downloadedvideos\Revhallyx - Nightmare [NA-NA].webm
[download] 100% of    3.25MiB in 00:00:00 at 7.39MiB/s   
Error during conversion: Command '['ffmpeg', '-i', '"Revhallyx - Nightmare [NA-NA].webm"', '-vn', '-ab', '192k', '-ar', '44100', '-y', '"Revhallyx - Nightmare [NA-NA].mp3"']' returned non-zero exit status 4294967274.


In [2]:
# pip install yt_dlp
# pip install https://github.com/seproDev/yt-dlp-ChromeCookieUnlock/archive/main.zip
from yt_dlp.utils import download_range_func
import yt_dlp
import json


ydl_opts = {
    "paths": {"home": "downloadedvideos/"},
    "format_sort": ["res:1080"],
    # 'format': 'bv',
    "outtmpl": "%(title)s.%(ext)s",
    "overwrite": True,
    "verbose": True,
    "cachedir": False,
    "forceurl": True,
    "proxy": "177.234.209.86:999",
    "skipdownload": True
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    info = ydl.extract_info("https://www.youtube.com/watch?v=jBlT4D01YOU")
    # print(json.dumps(info, indent=4))
    # print(json.dumps(ydl.extract_info('https://www.youtube.com/watch?v=Zdk6sVYZABE', download=False),indent=4))
    # ydl.download(['https://www.youtube.com/watch?v=jBlT4D01YOU'])

[debug] Encodings: locale cp1252, fs utf-8, pref cp1252, out UTF-8 (No VT), error UTF-8 (No VT), screen UTF-8 (No VT)
[debug] yt-dlp version stable@2025.01.26 from yt-dlp/yt-dlp [3b4531934] (pip) API
[debug] params: {'paths': {'home': 'downloadedvideos/'}, 'format_sort': ['res:1080'], 'outtmpl': '%(title)s.%(ext)s', 'overwrite': True, 'verbose': True, 'cachedir': False, 'forceurl': True, 'proxy': '177.234.209.86:999', 'compat_opts': set(), 'http_headers': {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/95.0.4638.17 Safari/537.36', 'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8', 'Accept-Language': 'en-us,en;q=0.5', 'Sec-Fetch-Mode': 'navigate'}}
[debug] Python 3.10.6 (CPython AMD64 64bit) - Windows-10-10.0.19045-SP0 (OpenSSL 1.1.1n  15 Mar 2022)
[debug] exe versions: ffmpeg 2024-05-15-git-7b47099bc0-essentials_build-www.gyan.dev (setts), ffprobe 2024-05-15-git-7b47099bc0-essentials_build-www.gyan.dev
[d

[youtube] Extracting URL: https://www.youtube.com/watch?v=jBlT4D01YOU
[youtube] jBlT4D01YOU: Downloading webpage
[youtube] jBlT4D01YOU: Downloading tv client config
[youtube] jBlT4D01YOU: Downloading player c8dbda2a
[youtube] jBlT4D01YOU: Downloading tv player API JSON
[youtube] jBlT4D01YOU: Downloading ios player API JSON


[debug] [youtube] Decrypted nsig _e9coEYs77C52qX => 1POM0xyYprey4g
[debug] [youtube] Decrypted nsig 8iVwF5QG7K0b2Eg => 1ACqQRbAk5s5mg
[debug] [youtube] jBlT4D01YOU: ios client https formats require a GVS PO Token which was not provided. They will be skipped as they may yield HTTP Error 403. You can manually pass a GVS PO Token for this client with --extractor-args "youtube:po_token=ios.gvs+XXX". For more information, refer to  https://github.com/yt-dlp/yt-dlp/wiki/PO-Token-Guide . To enable these broken formats anyway, pass --extractor-args "youtube:formats=missing_pot"


[youtube] jBlT4D01YOU: Downloading m3u8 information


[debug] Sort order given by user: res:1080
[debug] Sort order given by extractor: quality, res, fps, hdr:12, source, vcodec, channels, acodec, lang, proto
[debug] Formats sorted by: hasvid, ie_pref, res:1080(1080.0), quality, fps, hdr:12(7), source, vcodec, channels, acodec, lang, proto, size, br, asr, vext, aext, hasaud, id
[debug] Default format spec: bestvideo*+bestaudio/best


...

[download] Destination: downloadedvideos\Genshin Impact 3.3 - C0 Ganyu clear 12-2-2 Golden Wolflord in 17s.f136.mp4
[download] 100% of    5.85MiB in 00:00:15 at 379.66KiB/s 


[debug] Invoking http downloader on "https://rr5---sn-pm2-aume.googlevideo.com/videoplayback?expire=1740426740&ei=lHm8Z8zaFvS0zN0Py_L2qAY&ip=177.234.209.82&id=o-AOvipN3R3lAcf5kHZcQfQ7W9-CNd3Y29A2DExhtSMc16&itag=251&source=youtube&requiressl=yes&xpc=EgVo2aDSNQ%3D%3D&met=1740405140%2C&mh=wT&mm=31%2C29&mn=sn-pm2-aume%2Csn-cvb7lnls&ms=au%2Crdu&mv=m&mvi=5&pl=27&rms=au%2Cau&initcwndbps=1517500&bui=AUWDL3wKh0L8Lk0lbAKYUXUsMRQGrIlwFE3NDFE9Bo9Jwbf3NBB2hI77r1EKv5KO3uuj2NslD1WJmBXP&spc=RjZbSUc6yT6CAcTJd6q8Sond7jOOsHx2nP8f-eNv-jx9L_S1m1bmTjAd8k2Ypasitwfo4Q&vprv=1&svpuc=1&mime=audio%2Fwebm&ns=Xl3MDxpOd6wa3e6cDTxEiVQQ&rqh=1&gir=yes&clen=349634&dur=25.561&lmt=1671575803185207&mt=1740404711&fvip=1&keepalive=yes&lmw=1&fexp=51326932&c=TVHTML5&sefc=1&txp=5318224&n=1ACqQRbAk5s5mg&sparams=expire%2Cei%2Cip%2Cid%2Citag%2Csource%2Crequiressl%2Cxpc%2Cbui%2Cspc%2Cvprv%2Csvpuc%2Cmime%2Cns%2Crqh%2Cgir%2Cclen%2Cdur%2Clmt&sig=AJfQdSswRQIgZdTndgIkAVk8rI99haSOvM8-aONE7hiXa2Ydyxo6_lICIQChCPbQ3BPVGt7ICIBugGcSew7CyQtdVW

[download] Destination: downloadedvideos\Genshin Impact 3.3 - C0 Ganyu clear 12-2-2 Golden Wolflord in 17s.f251.webm
[download] 100% of  341.44KiB in 00:00:06 at 51.81KiB/s  
[Merger] Merging formats into "downloadedvideos\Genshin Impact 3.3 - C0 Ganyu clear 12-2-2 Golden Wolflord in 17s.mkv"


[debug] ffmpeg command line: ffmpeg -y -loglevel repeat+info -i "file:downloadedvideos\Genshin Impact 3.3 - C0 Ganyu clear 12-2-2 Golden Wolflord in 17s.f136.mp4" -i "file:downloadedvideos\Genshin Impact 3.3 - C0 Ganyu clear 12-2-2 Golden Wolflord in 17s.f251.webm" -c copy -map 0:v:0 -map 1:a:0 -movflags +faststart "file:downloadedvideos\Genshin Impact 3.3 - C0 Ganyu clear 12-2-2 Golden Wolflord in 17s.temp.mkv"


Deleting original file downloadedvideos\Genshin Impact 3.3 - C0 Ganyu clear 12-2-2 Golden Wolflord in 17s.f136.mp4 (pass -k to keep)
Deleting original file downloadedvideos\Genshin Impact 3.3 - C0 Ganyu clear 12-2-2 Golden Wolflord in 17s.f251.webm (pass -k to keep)


In [ ]:
"""
Proof of concept way to get cookies from chrome on Windows.. even when they're locked.
Does not require admin rights.
Includes a pure-python version of release_file_lock from:
https://github.com/thewh1teagle/rookie/blob/02995bbbb692f775e12368e7fb2b728775c88ddd/rookie-rs/src/winapi.rs#L63
(C) - MIT License 2023 - Charles Machalow
"""

import os
from ctypes import windll, byref, create_unicode_buffer, pointer, WINFUNCTYPE
from ctypes.wintypes import DWORD, WCHAR, UINT
import browser_cookie3  # pip install browser-cookie3
import backoff  # pip install backoff

ERROR_SUCCESS = 0
ERROR_MORE_DATA = 234
RmForceShutdown = 1

cookies_path = os.path.expandvars(
    r"%LOCALAPPDATA%\Google\Chrome\User Data\Default\Network\Cookies"
)

rstrtmgr = windll.LoadLibrary("Rstrtmgr")


@WINFUNCTYPE(None, UINT)
def callback(percent_complete: UINT) -> None:
    print(f"Unlocking file status: {percent_complete}% done")


def unlock_cookies():
    session_handle = DWORD(0)
    session_flags = DWORD(0)
    session_key = (WCHAR * 256)()

    result = DWORD(
        rstrtmgr.RmStartSession(byref(session_handle), session_flags, session_key)
    ).value

    if result != ERROR_SUCCESS:
        raise RuntimeError(f"RmStartSession returned non-zero result: {result}")

    try:
        result = DWORD(
            rstrtmgr.RmRegisterResources(
                session_handle,
                1,
                byref(pointer(create_unicode_buffer(cookies_path))),
                0,
                None,
                0,
                None,
            )
        ).value

        if result != ERROR_SUCCESS:
            raise RuntimeError(
                f"RmRegisterResources returned non-zero result: {result}"
            )

        proc_info_needed = DWORD(0)
        proc_info = DWORD(0)
        reboot_reasons = DWORD(0)

        result = DWORD(
            rstrtmgr.RmGetList(
                session_handle,
                byref(proc_info_needed),
                byref(proc_info),
                None,
                byref(reboot_reasons),
            )
        ).value

        if result not in (ERROR_SUCCESS, ERROR_MORE_DATA):
            raise RuntimeError(f"RmGetList returned non-successful result: {result}")

        if proc_info_needed.value:
            result = DWORD(
                rstrtmgr.RmShutdown(session_handle, RmForceShutdown, callback)
            ).value

            if result != ERROR_SUCCESS:
                raise RuntimeError(
                    f"RmShutdown returned non-successful result: {result}"
                )
        else:
            print("File is not locked")
    finally:
        result = DWORD(rstrtmgr.RmEndSession(session_handle)).value

        if result != ERROR_SUCCESS:
            raise RuntimeError(f"RmEndSession returned non-successful result: {result}")


# Use backoff here since there is a race condition between unlocking the file and reading it.
# Technically we're killing a process within chrome that holds the lock. Chrome can/will restart it,
# .. so we have to fetch cookies before it re-locks the file. Generally on my system we get them
# .... though one time we didn't. I think it has to do with opening a new tab.. idk. Maybe not necessary?
@backoff.on_exception(backoff.constant, PermissionError, max_tries=5)
def fetch_cookies():
    unlock_cookies()
    return browser_cookie3.chrome(cookies_path)


cookies = fetch_cookies()
print(len(cookies))